# Imports

In [1]:
import numpy as np
from bertopic import BERTopic
from transformers.pipelines import pipeline
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer


d:\DESIGN\ANACONDA\envs\bertopic_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load data

In [ ]:
with open("../data/02_cleaned/cleaned_responses_397.txt", "r", encoding="utf-8") as file:
    docs = file.readlines()

print("Number of documents:", len(docs))
print("First document preview:", docs[0])

vectorizer_model = None


Number of documents:  397
First document preview:  买错衣服尺码的时候 换货很麻烦



# Build model

In [ ]:
embedding_model = pipeline("feature-extraction", model="bert-base-chinese")

embeddings = np.load("../data/02_bertopic_outputs/data_emb.npy")
print("Embedding shape:", embeddings.shape)

umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=30
)

hdbscan_model = HDBSCAN(
    min_cluster_size=15,
    min_samples=5,
    metric="euclidean"
)

vectorizer_model = CountVectorizer()

topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
)

Some weights of the model checkpoint at bert-base-chinese were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.bias', 'cls.predictions.decoder.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Embedding shape: (397, 768)


In [6]:
topics, probs = topic_model.fit_transform(docs, embeddings=embeddings)
topic_model.get_topic_info()


,Topic,Count,Name,Representation,Representative_Docs
0,-1,103,-1_卫生问题_不卫生_去试衣间要排长队_卫生,"[卫生问题, 不卫生, 去试衣间要排长队, 卫生, 直接虚拟上身查看效果就非常好, 能尝试不...","[卫生问题\n, 卫生问题\n, 卫生问题\n]"
1,0,71,0_换季时整理衣柜的崩溃瞬间_衣服太多_总觉得衣柜满了但没衣服穿_衣柜满了,"[换季时整理衣柜的崩溃瞬间, 衣服太多, 总觉得衣柜满了但没衣服穿, 衣柜满了, 木质衣柜味...",[衣服太多，喜欢的款式太多，导致衣柜塞不下，换季收拾很崩溃，并且“一次性”的衣服太多但是均价...
2,1,54,1_无限的风格尝试_经济实惠_环保不浪费_极致的便捷,"[无限的风格尝试, 经济实惠, 环保不浪费, 极致的便捷, 无限风格尝试, 在这种状态下, ...",[通过色系，款式，分类服装，能确保在第一时间准确找到想要的搭配，依据种类分区放理，例如大衣外...
3,2,52,2_卫生问题_营业时间短_不准确_环境问题,"[卫生问题, 营业时间短, 不准确, 环境问题, 没有适合的尺码, 残留污渍, 某些衣服穿率...","[不干净卫生，营业时间短，排队长，试衣人数多，衣物清洗时间不足\n, 卫生问题，好几个人穿过..."
4,3,47,3_试衣间必须干净_灯光明亮_服务站的位置必须在宿舍楼下_app上库存必须实时,"[试衣间必须干净, 灯光明亮, 服务站的位置必须在宿舍楼下, app上库存必须实时, 有很多...","[试衣间必须干净，灯光明亮，app上的库存是实时的，归还流程必须简单\n, 试衣间必须干净、..."
5,4,30,4_空间大_试衣间干净_不想排队_有网上链接可预约,"[空间大, 试衣间干净, 不想排队, 有网上链接可预约, 数量多, 有全身镜, 实惠容量大,...","[看重方便 舒适 样式\n, 数量多，空间大\n, 衣服干净 种类多 空间大 服务便利\n]"
6,5,21,5_离宿舍近_环境干净_试衣间干净_归还简单,"[离宿舍近, 环境干净, 试衣间干净, 归还简单, 服务站距离宿舍近, 在宿舍楼下, 干净,...","[离宿舍近，干净，有良好的看管（不要把我衣服弄丢和被人家拿错），实时\n, 无限的风格尝试，..."
7,6,19,6_干净_不干净_清洁消毒功能_开心,"[干净, 不干净, 清洁消毒功能, 开心, 干净与安全, 尺寸不对, 宿舍下, 在需要的时候...","[干净\n, 干净\n, 干净\n]"


# Save clustering results

In [ ]:
topic_docs = topic_model.get_document_info(docs)
topic_docs.to_csv("../data/02_bertopic_outputs/clustering_results.csv", index=False)


# Visualization

In [9]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

topic_info = topic_model.get_topic_info()

valid_topic_info = topic_info[topic_info["Topic"] != -1].copy()

topic_ids = valid_topic_info["Topic"].tolist()[:8]

print("Topic IDs used for plotting:", topic_ids)

n_words = 5

cols = 2
rows = math.ceil(len(topic_ids) / cols)

translation_map = {
    "\u6362\u5b63\u65f6\u6574\u7406\u8863\u67dc\u7684\u5d29\u6e83\u77ac\u95f4": "Seasonal wardrobe sorting",
    "\u8863\u670d\u592a\u591a": "Excessive clothing",
    "\u8863\u67dc\u6ee1\u4e86": "Overcrowded wardrobe",
    "\u603b\u89c9\u5f97\u8863\u67dc\u6ee1\u4e86\u4f46\u6ca1\u8863\u670d\u7a7f": "Full closet but nothing to wear",
    "\u8863\u67dc\u592a\u5c0f": "Limited wardrobe space",

    "\u65e0\u9650\u7684\u98ce\u683c\u5c1d\u8bd5": "Unlimited style trials",
    "\u7ecf\u6d4e\u5b9e\u60e0": "Cost-effectiveness",
    "\u5728\u8fd9\u79cd\u72b6\u6001\u4e0b": "Under such conditions",
    "\u65e0\u9650\u98ce\u683c\u5c1d\u8bd5": "Unlimited style trials",
    "\u73af\u4fdd\u4e0d\u6d6a\u8d39": "Environmental sustainability",

    "\u8bd5\u8863\u95f4\u5fc5\u987b\u5e72\u51c0": "Clean fitting room",
    "\u706f\u5149\u660e\u4eae": "Bright lighting",
    "\u670d\u52a1\u7ad9\u7684\u4f4d\u7f6e\u5fc5\u987b\u5728\u5bbf\u820d\u697c\u4e0b": "Service point near dormitory",
    "app\u4e0a\u5e93\u5b58\u5fc5\u987b\u5b9e\u65f6": "Real-time app inventory",
    "\u73af\u5883\u80fd\u591f\u5e72\u51c0\u660e\u4eae": "Clean and bright environment",

    "\u536b\u751f\u95ee\u9898": "Hygiene concerns",
    "\u8425\u4e1a\u65f6\u95f4\u77ed": "Limited service hours",
    "\u6709\u6c61\u6e0d": "Stains",
    "\u8863\u7269\u6e05\u6d17\u65f6\u95f4\u4e0d\u8db3": "Insufficient cleaning time",
    "\u8863\u7269\u5c11\u4e14\u5c3a\u7801\u4e0d\u5408\u9002": "Limited items and sizes",

    "\u5e72\u51c0": "Cleanliness",
    "\u4e0d\u5e72\u51c0": "Uncleanliness",
    "\u4f1a\u4e0d\u4f1a\u4e0d\u5e72\u51c0": "Perceived uncleanliness",
    "\u5e72\u51c0\u4e0e\u5b89\u5168": "Cleanliness and safety",
    "\u5bbf\u820d\u4e0b": "Near dormitories",

    "\u7a7a\u95f4\u5927": "Large storage space",
    "\u8bd5\u8863\u95f4\u5e72\u51c0": "Clean fitting room",
    "\u6709\u5168\u8eab\u955c": "Full-length mirror",
    "\u6570\u91cf\u591a": "Sufficient quantity",
    "\u968f\u5904\u53ef\u89c1": "High accessibility",

    "\u6362\u5b63\u65f6\u8981\u6574\u7406\u8863\u67dc": "Seasonal wardrobe management",
    "\u8863\u670d\u635f\u574f\u548c\u6c61\u6e0d": "Clothing damage and stains",
    "\u8863\u670d\u4e0d\u5c0f\u5fc3\u5f04\u810f": "Accidental clothing stains",
    "\u7a7f\u8863\u670d\u5f88\u9ebb\u70e6": "Dressing inconvenience",
    "\u6d17\u8863\u670d\u592a\u70e6\u4e86": "Laundry burden",

    "\u73af\u5883\u5e72\u51c0": "Clean environment",
    "\u670d\u52a1\u7ad9\u8ddd\u79bb\u5bbf\u820d\u8fd1": "Service point near dormitory",
    "\u5f52\u8fd8\u7b80\u5355": "Easy return process",
    "\u79bb\u5bbf\u820d\u8fd1": "Proximity to dormitory"
}

def translate_label(word):
    return translation_map.get(str(word), str(word))

def wrap_label(text, max_chars=22):
    text = str(text)
    words = text.split()

    if len(words) <= 1:
        return text

    lines = []
    current = ""

    for word in words:
        candidate = (current + " " + word).strip()
        if len(candidate) <= max_chars:
            current = candidate
        else:
            if current:
                lines.append(current)
            current = word

    if current:
        lines.append(current)

    return "<br>".join(lines)

fig = make_subplots(
    rows=rows,
    cols=cols,
    subplot_titles=[f"Topic {i}" for i in topic_ids],
    horizontal_spacing=0.24,
    vertical_spacing=0.10
)

colors = [
    "#D55E00", "#0072B2", "#CC79A7", "#E69F00",
    "#56B4E9", "#009E73", "#F0E442", "#999999"
]

for idx, topic_id in enumerate(topic_ids):
    row = idx // cols + 1
    col = idx % cols + 1

    topic_words = topic_model.get_topic(topic_id)

    if not topic_words:
        print(f"Skipping invalid topic: {topic_id}")
        continue

    topic_words = topic_words[:n_words]

    original_words = [word for word, score in topic_words]
    scores = [score for word, score in topic_words]

    words_en = [
        wrap_label(translate_label(word), max_chars=22)
        for word in original_words
    ]

    fig.add_trace(
        go.Bar(
            x=scores[::-1],
            y=words_en[::-1],
            orientation="h",
            marker=dict(color=colors[idx % len(colors)]),
            showlegend=False,
            customdata=original_words[::-1],
            hovertemplate=(
                "English label: %{y}<br>"
                "Original label: %{customdata}<br>"
                "Score: %{x:.4f}<extra></extra>"
            )
        ),
        row=row,
        col=col
    )

fig.update_layout(
    width=1300,
    height=max(900, rows * 420),
    title=dict(
        text="Topic Word Scores",
        x=0.5,
        xanchor="center",
        font=dict(size=26)
    ),
    font=dict(
        family="Arial",
        size=13
    ),
    margin=dict(
        l=320,
        r=80,
        t=120,
        b=80
    ),
    plot_bgcolor="white",
    paper_bgcolor="white"
)

fig.update_xaxes(
    showgrid=True,
    gridcolor="#EAECEF",
    zeroline=False,
    tickfont=dict(size=11),
    title_text=""
)

fig.update_yaxes(
    automargin=True,
    tickfont=dict(size=12),
    title_text=""
)

fig.update_annotations(
    font=dict(size=16)
)

fig.show()

fig.write_image("topic_word_scores_english.png", scale=3)
fig.write_image("topic_word_scores_english.svg")
fig.write_image("topic_word_scores_english.pdf")


Topic IDs used for plotting: [0, 1, 2, 3, 4, 5, 6]


ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido
